In [ ]:
import os

# =====================================================
# 📁 데이터 파일 경로 설정 (본인 환경에 맞게 수정하세요)
# =====================================================
BASE_DIR = os.getcwd()  # 현재 작업 디렉토리 (노트북이 있는 폴더)
print(f'작업 디렉토리: {BASE_DIR}')


# ◆ 0. 샘플 데이터 불러오기

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(os.path.join(BASE_DIR, 'data.csv'))
df

# ◆ 1. 데이터 전처리하기
* 1.1. 특수문자 및 숫자 제거
* 1.2. 의미없는 짧은 글 제거

### 1.1. 특수문자 및 숫자 제거
* 정규표현식을 사용해봅시다 😸

In [ ]:
# 라이브러리/모듈 import 하기

import re # 정규표현식을 사용하기 위한 파이썬 기본 내장 라이브러리
from tqdm import tqdm # 반복문이 얼마나 진행됐는지 진행률 바로 보여주는 라이브러리

In [ ]:
# 텍스트마이닝 복습1) 허용하지 않는 문자를 공백으로 바꾸기
# re.compile() : 정규표현식 패턴을 미리 준비해두는 함수
# re.sub() : 문자열에서 특정 패턴을 찾아서 다른 문자로 바꿔주는 함수

# 영문자, 한글, 공백, '.', '!', '?' 를 제외한 모든 문자 바꾸기
pattern = re.compile(r'[^a-zA-Z가-힣\s\.\!\?]')
string = re.sub(pattern, ' ', "DCX_프로세스중입니다           화이팅~")
print(string)

In [ ]:
# 텍스트마이닝 복습2) 연속된 공백을 하나로 합치기
pattern2 = re.compile(r'\s+')
result = re.sub(pattern2, ' ', string)
print(result)

In [ ]:
# 전처리 코드를 하나의 함수로 만들기 -> 재사용이 가능하다!
# apply() : 하나의 행/열 값에 함수를 일괄 적용해줌

def re_pattern(string):
  pattern = re.compile(r'[^a-zA-Z가-힣\s\.\!\?]')
  string = re.sub(pattern, ' ', string)

  pattern2 = re.compile(r'\s+')
  result = re.sub(pattern2, ' ', string)

  return result

In [ ]:
# Review 열 전체에 적용해서, df에 새로운 열로 추가해보기

df['re_review'] = df['Review'].apply(lambda x : re_pattern(x)) # re_pattern(x) => x를 받아서, re_pattern함수를 실행해주세요 ~
df

# apply() : 열의 모든 값에 함수를 한줄씩 적용하는 메서드
# lambda : 1회용으로 쓰는 간의 함수, 간단한 처리를 1줄로 쓸 때 사용함

### 1.2. 의미없는 짧은 글 제거
* 15자 미만인 리뷰를 제거해봐요 😸

In [ ]:
# 리뷰를 한 줄씩 확인하면서 15자 미만이면 해당 행을 직접 삭제 -- enumerate 활용

for n, i in enumerate(df['Review']):
  if len(i) < 15:
    df = df.drop([n])

df

In [ ]:
# 행 삭제 후 뒤죽박죽 된 인덱스 0부터 다시 정렬

df = df.reset_index(drop=True) # 기본값이 False, 기존 인덱스를 버리고, 0부터 새로 번호 매기고 싶음
df

`enumerate()` : 반복 가능한 객체를 입력 받아, 각 요소에 대해 인덱스와 값을 함께 반환하는 파이썬 내장 함수입니다.


```
    fruits = ['apple', 'banana', 'cherry']
    for i, fruit in enumerate(fruits):
        print(i, fruit)
```
    
  즉, `enumerate(fruits)`는 내부적으로 다음과 같은 튜플을 생성합니다 :
    
  ```
  [(0, 'apple'), (1, 'banana'), (2, 'cherry')]
  ```
    
  ➡️ 반복문을 돌면서, 지금 몇 번 째 반복인지’를 자동으로 알려줄 수 있습니다 (i 값)

# ◆ 2. 데이터 형태소 분리하기
(제공되는 한국어 불용어 파일 사용: ko-stopwords.csv)
* 2.1 불용어적용 및 형태소 분리
* 2.2 데이터 프레임에 추가

In [ ]:
!pip install konlpy

In [ ]:
# Okt(Open Korean Text)는 트위터가 만든 한국어 형태소 분석기로,
# 한국어 문장을 단어 단위로 쪼개고 각 단어의 품사(명사/동사/형용사 등)를 태깅해주는 도구입니다.

from konlpy.tag import Okt
okt = Okt()                 # 형태소 분석기 객체를 생성

### 2.1. 불용어적용 및 형태소 분리

In [ ]:
# 불용어 파일 가져오기
stopwords_df = pd.read_csv(os.path.join(BASE_DIR, 'ko-stopwords.csv'))
stopwords_df

In [ ]:
stopwords = set(stopwords_df['stopwords'])

# 왜? -> set(집합)으로 변환을 해주면 연산속도가 리스트보다 훨씬 빨라요

In [ ]:
# 형태소를 분석을 하고, 불용어도 제거
# 함수형태로 정의하면 재사용성 Up!

def okt_pos_tagging(string):
  pos_word = okt.pos(string, stem=True, norm=True)
  # okt.pos() : 문자열을 형태소 단위로 분리해주고, 품사를 태깅해주는 함수
  # (단어, 품사태그) 형태로 튜플 리스트로 반환을 해줌
  # stem=True : 활용형을 기본형으로 변환해주는 옵션 ('먹었다', '먹고', -> '먹다')
  # norm=True : 비표준 표현을 정규화해주는 옵션 ('ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ' -> 'ㅋㅋㅋ')

  # 형태소 분석한 값을 불용어 처리하기!
  # 리스트 컴프리헨션 : 리스트를 한줄로 간결하게 만드는 문법
  # [표현식 for 변수 in 반복대상]
  result = [word for word, tag in pos_word if word not in stopwords if tag in {'Noun', 'Adjective', 'Verb'}]
    # pos_word의 (단어, 품사태그) 쌍을 순회하면서, 불용어 목록에 없는 단어, 명사/형용사/동사 품사만
  return result

In [ ]:
# 첫번째 리뷰에 함수 적용해서 결과 확인하기
okt_pos_tagging(df['Review'][0])

### 2.2 데이터 프레임에 추가

In [ ]:
df

In [ ]:
# progress_apply() : apply()와 동일하지만 진행률 바를 함께 표시
# 형태소 분석은 시간이 오래 걸리기 때문에 유용

tqdm.pandas()

df['tagged_review'] = df['re_review'].progress_apply(lambda x : okt_pos_tagging(x))
df

# ◆ 3. 벡터화
* 3.1 doc2vec 준비(문서의 순서 매기기)
* 3.2 doc2vec 학습시키기
* 3.3 벡터 값 데이터 프레임에 추가

In [ ]:
!pip install gensim

In [ ]:
import gensim # 자연어 처리 및 토픽 모델링을 위한 라이브러리
from gensim.models.doc2vec import TaggedDocument # gensim의 doc2vec 모듈에서 문서에 태그(레이블)를 함께 저장하기 위한 TaggedDocument 클래스
from gensim.models import Doc2Vec # 문서 임베딩 학습을 위한 Doc2Vec 모델 클래스

### 3.1 doc2vec 준비 : 각 문서에 고유 ID(태그)를 붙인 TaggedDocument 객체 리스트 생성
* word2vec은 단어 하나를 하나의 vector화 (단어 1개 → 벡터 1개)
* doc2vec은 문서 하나를 하나의 vector화 (문서 1개 → 벡터 1개; 문서 전체의 의미를 하나의 숫자 배열로 표현)

TaggedDocument란? Doc2Vec에게 학습 데이터를 넘겨줄 때 사용하는 전용 그릇(객체)입니다.

```
TaggedDocument(
    words = ["냉장고", "소음", "크다"],  # 단어 리스트
    tags  = ["document0"]               # 이 문서의 고유 ID
)
```

words: 분석할 단어 목록["냉장고", "소음", "크다"]
tags: 이 문서를 구별하는 ID["document0"]

왜 일반 리스트가 아니라 TaggedDocument를 쓸까요?


```
# 일반 리스트만 있는경우
["냉장고", "소음", "크다"]   # "이게 몇 번 문서야?" → Doc2Vec이 모름

# TaggedDocument를 쓰면
TaggedDocument(
    words=["냉장고", "소음", "크다"],
    tags=["document0"]               # "0번 문서야!" → Doc2Vec이 기억
)
```

Doc2Vec은 단어의 의미뿐 아니라 "이 단어들이 어느 문서에 속하는가" 도 함께 학습합니다.
그래서 반드시 문서 ID(tags)가 붙어 있어야 합니다.

In [ ]:
# TaggedDocument 만들어보기!

tagged_corpus_list = [] # TaggedDoument 객체들을 담을 빈 리스트

for n, i in enumerate(df['tagged_review']):
  tag = 'document{}'.format(n)
  tagged_corpus_list.append(TaggedDocument(tags=[tag], words=i)) # TaggedDocument : Doc2Vec 학습용 객체
  # TaggedDocument(words['시스템', '에어컨']..., tags=['document0'])

In [ ]:
tagged_corpus_list[0]

### 3.2 doc2vec 학습시키기

####  하이퍼파라메터 설명
**1. vector_size → 문서 임베딩 차원 수**
* 소량 데이터(문서 수 몇 천 단위): 100 ~ 200(권장)
* 중간 이상 데이터(문서 수 수만 이상): 200 ~ 400(권장)
* 너무 크게 잡으면 학습 시간이 증가하고, 데이터가 적으면 과적합 위험이 있습니다.
* 처음에는 200 또는 300 정도에서 시작 후, 성능 보고 조정하는 패턴을 추천합니다.

**2. alpha, min_alpha → 초기 학습률 / 마지막 학습률**
*  alpha: 흔히 0.025 ~ 0.05 정도에서 시작하며, 값이 클수록 빠르게 학습하지만 불안정해질 수 있습니다.
* min_alpha: 학습 후반에 내려갈 학습률로, 보통 0.0001 ~ 0.001 정도로 두는 경우가 많습니다.
* 실무에서 자주 쓰는 조합 예시: alpha=0.025, min_alpha=0.0001
* 너무 큰 alpha → 학습이 요동
* min_alpha를 너무 크게 두면 → 마지막까지 파라미터가 너무 많이 움직임

**3. window → 주변 단어를 몇 개까지 컨텍스트로 볼지**
* 리뷰/짧은 문장 위주 텍스트: 3 ~ 5(권장)
* 긴 문서, 문맥 넓게 보고 싶을 때: 5 ~ 10(권장)
* 너무 작으면 문맥 정보 부족, 너무 크면 관계 없는 단어까지 섞여 노이즈 증가합니다.
* 지금 설정한 window=3은 짧은 리뷰 기준으로는 무난한 값으로, 좀 더 넓게 의미를 보고 싶다면 5 정도도 많이 사용합니다.

**4. min_count → 최소 몇 번 이상 등장한 단어만 학습에 포함할지**
* 일반적인 텍스트 마이닝: 2 ~ 5(권장)
* 데이터가 매우 적은 경우 불가피하게 1을 쓰기도 합니다. 다만 노이즈가 많아질 수 있어요! (오타, 고유명사, 희귀 단어까지 모두 포함하기 때문에)
* 리뷰 데이터가 수천 개 이상이라면: min_count=2 또는 3 정도로 시작하는 경우가 많습니다.

**5. dm → 학습 방식 선택 (Doc2Vec 알고리즘 타입)**
* dm=1 : DM(Distributed Memory) 방식으로 문맥 + 문서 벡터를 함께 사용합니다.
  * 문서 의미를 안정적으로 잡는 데 자주 사용합니다.
* dm=0 : DBOW(Distributed Bag of Words) 방식으로, word2vec의 Skip-gram과 비슷합니다.
  * 종종 더 빠르고, word 벡터 성능이 좋은 경우 있습니다.
  * 고급 튜닝에서는 dm=1 모델, dm=0 모델 둘 다 학습한 뒤, 벡터를 이어 붙이거나(concatenate) 둘 중 더 좋은 쪽을 선택하기도 합니다.

In [ ]:
# 모델 생성하기

model_doc2vec = Doc2Vec(
    vector_size = 300,
    alpha = 0.025,
    min_alpha = 0.01,
    window = 3,
    min_count = 1,
    dm = 1
)

In [ ]:
# 1) 단어 사전 구축 (학습 전 준비 단계) -> 학습 전에 어떤 단어들이 우리 데이터에 존재하는지 Doc2Vec 모델이 알아야 학습

model_doc2vec.build_vocab(tagged_corpus_list)
# tagged_corpus_list 전체를 훑어서 등장잔 단어 목록과 빈도수를 저장

In [ ]:
# 2) 모델 학습

model_doc2vec.train(
    tagged_corpus_list,
    total_examples = model_doc2vec.corpus_count, # 전체 문서 수
    epochs = 100
)

# tagged_corpus_list 전체를 100번 반복해서 읽으면서, 각 문서의 벡터를 업데이트

In [ ]:
# 3) 학습 결과 확인하기
model_doc2vec.dv['document0']

# TaggedDoument: word['', '', ..], tags[document(i)]

# [0.21, -0.11, .. ] 300차원의 배열

### 3.3 벡터 값 데이터 프레임에 추가

In [ ]:
df

In [ ]:
# df에 문서별 벡터를 꺼내서 리스트로 만들고 추가하기

vector_list = []

for i in range(len(df)):
  doc2vec = model_doc2vec.dv[f'document{i}']
  vector_list.append(doc2vec)

In [ ]:
df['vector'] = vector_list
df.head()

# ◆ 4. 병합 계층적 클러스터링
* 4.1 ward 기준으로 덴드로그램 그려보기
* 4.2 실루엣 지수 확인해서 토픽 갯수 정하기
* 4.3 가장 적절한 클러스터링 갯수 df에 삽입

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage  # 계층적 군집 분석(linkage)과 시각화(dendrogram) 함수 (그림을 그려줄 수 있는 친구)
from matplotlib import pyplot as plt                     # 그래프 시각화 라이브러리

### 4.1 ward 기준으로 덴드로그램 그려보기
- 덴드로그램 : 문서들이 어떻게 묶이는지를 나무(tree) 형태로 보여주는 그래프입니다.
- 가지가 합쳐지는 높이(거리)가 클수록 두 군집이 서로 다르다는 뜻입니다.

In [ ]:
# ward : 클러스터를 합친 때 군집 내 분산이 최소가 되도록 묶는 방식
# 다른 수치 데이터에 비해서 분산으로 계산했을 때 클리어하게 나눠지는 경향

model_linkage = linkage(list(df['vector']), 'ward')

# 덴드로그램 그려보기
plt.figure(figsize=(10, 5))

dendrogram(
    model_linkage, # 만들어놓은 모델 삽입
    orientation = 'top', # 덴드로그램 방향을 설정하는 코드 -- top: 위에서 아래로 가지가 뻗어나간다
    distance_sort = 'descending', # 거리가 큰(덜 유사한) 병합부터 왼쪽에 표시
    show_leaf_counts = False # 각각 합쳐진 가지들(값) 몇개씩 합쳐졌는지 숫자로 표기해주는 것 -- True하면 지져분해진다!
)

plt.show()

### 4.2 실루엣 지수 확인해서 토픽 갯수 정하기
- 실루엣 지수: 군집화가 얼마나 잘 됐는지 평가하는 점수 (-1 ~ 1) 입니다.
- 1에 가까울수록 같은 군집끼리는 가깝고, 다른 군집과는 멀리 떨어져 있음을 의미합니다.

In [ ]:
from sklearn.metrics.cluster import silhouette_score    # 실루엣 점수 계산 함수
from sklearn.cluster import AgglomerativeClustering     # 병합형 계층 군집 알고리즘 (클러스터 분리를 할 때 사용함)

In [ ]:
# k=3일때 실루엣 지수 뽑아보기

cluster_model = AgglomerativeClustering(n_clusters = 3, linkage = 'ward')

In [ ]:
cluster_label = cluster_model.fit_predict(list(df['vector']))
cluster_label

In [ ]:
silhouette_s = silhouette_score(list(df['vector']), cluster_label)
silhouette_s

# 공간에 흩어져있는 벡터값들이 어떤 클러스터에 배당 되어있느니 보고
# 실루엣 점수를 계산을 해줌

In [ ]:
# for문 돌려보기! : 클러스터 수를 2~30개까지 바꿔가며 실루엣 점수를 계산해서 최적의 k값 탐구
# x축 : 클러스터 개수 (k), y축 : 실루엣 점수

n_cluster = []
clustering_score = []

for i in tqdm(range(2, 31)):
  cluster_model = AgglomerativeClustering(n_clusters = i, linkage = 'ward')
  cluster_label = cluster_model.fit_predict(list(df['vector']))
  score = silhouette_score(list(df['vector']), cluster_label)

  n_cluster.append(i)
  clustering_score.append(score)

clustering_score

In [ ]:
plt.plot(n_cluster, clustering_score)

In [ ]:
# 클러스터 수별 점수를 df로 뽑아보기

result = pd.DataFrame({'n_cluster': n_cluster, 'score': clustering_score})
result

### 4.3 가장 적절한 클러스터링 갯수 df에 삽입

In [ ]:
# 최종 결정한 클러스터 수로 최종 군집화 수행

cluster_model = AgglomerativeClustering(n_clusters = 4, linkage = 'ward')
cluster_label = cluster_model.fit_predict(list(df['vector'])) # cluster = 0, 1, 2, 3
# score = silhouette_score(list(df['vector']), cluster_label)
cluster_label

In [ ]:
df['actor_cluster'] = cluster_label
df.head()

# ◆ 5. 해석하기:TF-IDF
* 문서 내에서 어떤 단어가 얼마나 중요한지를 평가하는 데 사용되는 방법
→ 각 문서에서의 핵짐 주제열을 판단할 수 있는 빈도분석의 기법
* 5.1 TF-IDF 계산
* 5.2 데이터프레임으로 만들고 정렬하기

### 5.1 TF-idf 계산
* 각 클러스터 마다 tfidf가 높은 워드들 찾기
* 각 클러스터들을 하나의 문서로 가정하여 tf-idf 값 추출

In [ ]:
from collections import Counter                              # 각 클러스터 내 단어 빈도 등을 계산할 때, 원소의 출현 횟수를 손쉽게 세기 위한 Counter 클래스
import numpy as np                                           # 수치 연산, 배열 처리, 벡터/행렬 연산 등을 위해 numpy 라이브러리
from sklearn.feature_extraction.text import TfidfVectorizer # 텍스트 데이터를 TF-IDF(단어 빈도-역문서 빈도) 방식의 수치 벡터로 변환하기 위한 TfidfVectorizer 클래스

In [ ]:
df

In [ ]:
# TfidfVectorizer의 경우 형태로 분리된 단어들이 띄어쓰기로 구분된 문자열을 입력으로 받음
# 단어 리스트 [...] '단어 단어 단어 ...'

all_document = [] # 클러스터별로 합쳐진 하나의 큰 문자열을 담을 리스트
# 각 클러스터를 순회 tagged_review

for i in range(0, 4):
  pos_tagging = df[df['actor_cluster'] == i]['tagged_review'] # i번 클러스터에 속한 행만 필터링, tagged_review 선택

  document = '' # 해당 되는 클러스터의 모든 단어들을 이어붙일 빈 문자열

  for pos in pos_tagging:
    doc = ' '.join(pos) + ' ' # 단어 리스트로 변환 ('단어 단어 단어')
    document += doc # 클러스터 전체 문자열(누적)

  all_document.append(document)

In [ ]:
all_document[0][:100]

In [ ]:
len(all_document)

In [ ]:
# tfidfvectorizer 생성하기

vectorizer = TfidfVectorizer() # 변환기 객체 생성
tfidf_matrix = vectorizer.fit_transform(all_document) # fit_transform : 단어 사전을 학습(fit), TF-IDF 행렬 생성(transform) 동시 수행
# 결과 : (클러스터 수 x 전체 단어 수) 크기의 희소 행렬

In [ ]:
# 엑셀로 추출하기 위해 데이터프레임으로 만들기
# 컬럼값 추출하기

feature_name = vectorizer.get_feature_names_out()
feature_name

In [ ]:
tfidf_value = tfidf_matrix.toarray()

# tfidf 점수 추출 할 수 있음

In [ ]:
tfidf_value

### 5.2 데이터프레임으로 만들고 정렬하기

In [ ]:
tfidf_df = pd.DataFrame(tfidf_value, columns = feature_name)
tfidf_df

In [ ]:
# Transpose : 행렬 전환 가능

tfidf_df = tfidf_df.T # 행 <-> 열 변경
tfidf_df

In [ ]:
data = tfidf_df[0].sort_values(ascending=False) # 해당 클러스터의 단어별 점수를 높은 순으로 정렬
data.to_csv(os.path.join(BASE_DIR, 'cluster0_tfidf.csv'), encoding='utf-8-sig') # 한글이기 때문에 인코딩 필수

In [ ]:
for i in tfidf_df.columns:
  data = tfidf_df[i].sort_values(ascending=False) # 해당 클러스터의 단어별 점수를 높은 순으로 정렬
  data.to_csv(os.path.join(BASE_DIR, f'cluster{i}_tfidf.csv'), encoding='utf-8-sig') # 한글이기 때문에 인코딩 필수

In [ ]:
df

In [ ]:
# 군집화 결과 전체를 pickle로 저장
# pickle의 가장 큰 장점! : 데이터 형태 그대로 저장을 해준다
  # 단점 : 간혹다가다 데이터를 빠뜨리고 저장하는 경우가 있다 ㅠㅠ
  # -> 꼭 저장하고 다시 불러와서 잘 불러와지는지 확인을 해야합니다

import pickle

with open(os.path.join(BASE_DIR, 'actor_clustering_result.pkl'), 'wb') as f:
  pickle.dump(df, f)

In [ ]:
with open(os.path.join(BASE_DIR, 'actor_clustering_result.pkl'), 'rb') as f:
  df2 = pickle.load(f)

df2